In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.12.0+cpu
GPU available: False


In [3]:
# (A1) -->A tensor is a multi-dimensional array used for numerical computations. Unlike a Python list, tensors are optimized for fast mathematical operations and can run on GPUs. An image (pixels arranged in rows, columns, and color channels) is an everyday example of data stored as a tensor.
#(A2) -->Shape tells the size of a tensor in each dimension, ndim tells the number of dimensions, and dtype tells the type of data stored in the tensor (e.g., integer or float).
#(A3) -->We move tensors and models to a GPU to speed up computations. GPUs can process many operations in parallel, making training and inference much faster for large datasets and models.
#(A4) -->Training data is used to teach the model, while test data is used to evaluate its performance. They must be kept separate to check how well the model generalizes to unseen data.
#(A5) -->A model learns the best values of its weights and bias so that its predictions are as close as possible to the actual data.

In [4]:
#  Part B Tensor Basics
# (B1)
tensor = torch.arange(1, 11)
print("Tensor:", tensor)
print("Shape:", tensor.shape)
print("Number of dimensions:", tensor.ndim)
print("Data type:", tensor.dtype)

Tensor: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
Shape: torch.Size([10])
Number of dimensions: 1
Data type: torch.int64


In [5]:
# ------ B2-----
# Creates random tensors
A = torch.rand(3, 4)
B = torch.rand(4, 2)

# Matrix multiplication
C = torch.matmul(A, B)

# Prints tensors and result shape
print("Shape of A:", A.shape)
print("Shape of B:", B.shape)
print("Shape of result:", C.shape)

Shape of A: torch.Size([3, 4])
Shape of B: torch.Size([4, 2])
Shape of result: torch.Size([3, 2])


In [6]:
#------B3-------
# Assuming (3, 4) tensor is stored in A
print("Minimum:", A.min())
print("Maximum:", A.max())
print("Mean:", A.mean())
print("Sum:", A.sum())

Minimum: tensor(0.2402)
Maximum: tensor(0.9856)
Mean: tensor(0.7221)
Sum: tensor(8.6656)


In [7]:
#------B4------
tensor = torch.arange(0, 100, 10).reshape(2, 5)
print(tensor)
print(tensor.shape)

tensor([[ 0, 10, 20, 30, 40],
        [50, 60, 70, 80, 90]])
torch.Size([2, 5])


In [8]:
#--------B5-------
# B5 explanation: Why is setting a random seed useful?
# ans : Setting a random seed ensures reproducibility, meaning the same random numbers are generated each time the code is run.
torch.manual_seed(42)
tensor = torch.rand(2, 3)
print(tensor)

tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])


In [9]:
import torch

# The "true" values our model will try to discover
weight, bias = 0.7, 0.3

# Create the data
X = torch.arange(0, 1, 0.02).unsqueeze(1)   # inputs,  shape: (50, 1)
y = weight * X + bias                        # outputs, shape: (50, 1)

print("X shape:", X.shape, "| y shape:", y.shape)
print("First 5 X values:", X[:5].squeeze())
print("First 5 y values:", y[:5].squeeze())

X shape: torch.Size([50, 1]) | y shape: torch.Size([50, 1])
First 5 X values: tensor([0.0000, 0.0200, 0.0400, 0.0600, 0.0800])
First 5 y values: tensor([0.3000, 0.3140, 0.3280, 0.3420, 0.3560])


In [10]:
#-----C1------
# Split data: 80% train, 20% test
train_size = int(0.8 * len(X))
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 40
Test samples: 10


In [11]:
#---------C2-------
from torch import nn
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=1, out_features=1)
    def forward(self, x):
        return self.linear_layer(x)
model = LinearRegressionModel()
print(model)

LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


In [12]:
#-------C3-----
# Defines loss function
loss_fn = nn.L1Loss()
# Defines optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
print(loss_fn)
print(optimizer)

L1Loss()
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [13]:
#--------C4------
# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    # 1. Forward pass
    y_pred = model(X_train)
    # 2. Calculate loss
    loss = loss_fn(y_pred, y_train)
    # 3. Zero gradients
    optimizer.zero_grad()
    # 4. Backpropagation
    loss.backward()
    # 5. Update parameters
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch: {epoch} | Loss: {loss.item():.5f}")

Epoch: 0 | Loss: 0.26975
Epoch: 20 | Loss: 0.23556
Epoch: 40 | Loss: 0.21770
Epoch: 60 | Loss: 0.20654
Epoch: 80 | Loss: 0.19826
Epoch: 100 | Loss: 0.19088
Epoch: 120 | Loss: 0.18390
Epoch: 140 | Loss: 0.17692
Epoch: 160 | Loss: 0.16994
Epoch: 180 | Loss: 0.16305


In [14]:
#------C5-----
# Prints learned weight and bias
print("Learned weight:", model.linear_layer.weight.item())
print("Learned bias:", model.linear_layer.bias.item())

Learned weight: -0.07704499363899231
Learned bias: 0.6262807846069336


In [ ]:

# ----- C6 -----

# Making  predictions on the test set
model.eval()
with torch.inference_mode():
    y_preds = model(X_test)
print("Predictions:")
print(y_preds)


import matplotlib.pyplot as plt
model.eval()
with torch.inference_mode():
    y_preds = model(X_test)

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label="Training Data")
plt.scatter(X_test, y_test, label="Test Data")
plt.scatter(X_test, y_preds, label="Predictions")

plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()

Predictions:
tensor([[0.5646],
        [0.5631],
        [0.5616],
        [0.5600],
        [0.5585],
        [0.5569],
        [0.5554],
        [0.5539],
        [0.5523],
        [0.5508]])


In [ ]:
#-----D1---
#The most confusing thing was the PyTorch training loop. I understood it by practicing with examples and checking the documentation.
#-----D2----
#Backpropagation (loss.backward()) still feels a bit fuzzy because I'm still learning how gradients are calculated and used
#-----D3-----
#4/5 — I am comfortable creating, reshaping, and performing operations on tensors, but I need more practice with advanced tensor manipulations.